# UrduCap: run every experiment on a GPU

Runs on **Kaggle** (recommended: attach the *Flickr Image dataset* by `hsankesara`, so no
8 GB download; free T4 ×2 or P100, 30 h/week) or **Colab** (T4). Each step writes to `runs/`,
so it can be resumed in a new session.

Set the three paths in the next cell. `CAPTIONS` is the Urdu caption file:
the Karpathy-style `captions_30k.json` from 2024, or any `image_name|comment` table.

In [ ]:
import os
IMAGES = "/kaggle/input/flickr-image-dataset/flickr30k_images/flickr30k_images"  # Flickr30k JPEGs
CAPTIONS = "/kaggle/input/urdu-flickr30k/captions_30k.json"                       # Urdu captions
WORK = "/kaggle/working"                                                           # runs/ goes here
os.chdir(WORK)

In [ ]:
!git clone -q https://github.com/shelenderrkumar/Urdu-Image-Captioning.git repo
!pip install -q -e repo
os.chdir(f"{WORK}/repo")

## 1. Data audit and the image-level split

In [ ]:
!urducap audit --captions $CAPTIONS
# Standard Karpathy split, so numbers are comparable with the literature.
!wget -q -nc https://cs.stanford.edu/people/karpathy/deepimagesent/caption_datasets.zip && unzip -o -q caption_datasets.zip dataset_flickr30k.json
!urducap split --captions $CAPTIONS --karpathy dataset_flickr30k.json --out data/splits.json

## 2. Zero-shot vision-language model (no training)

In [ ]:
!urducap evaluate --zero-shot vlm --captions $CAPTIONS --images $IMAGES --batch-size 8 --beams 1 --out runs/vlm-zeroshot

## 3. The 2024 architecture, re-trained under the corrected protocol

ResNet-101 + attention GRU, the same sizes as in 2024.

In [ ]:
!urducap train --model legacy --captions $CAPTIONS --images $IMAGES --epochs 8 --batch-size 64 --lr 4e-4 --out runs/legacy
!urducap evaluate --run runs/legacy --captions $CAPTIONS --images $IMAGES --beams 1 --out runs/legacy

## 4. SigLIP 2 encoder + mBART-50 Urdu decoder

Trains the projector and the decoder layers (about 200 M parameters).

In [ ]:
!urducap train --model encdec --captions $CAPTIONS --images $IMAGES --epochs 3 --batch-size 32 --lr 5e-5 --out runs/encdec
!urducap evaluate --run runs/encdec --captions $CAPTIONS --images $IMAGES --out runs/encdec

## 5. Vision-language model with LoRA

Qwen3-VL-2B-Instruct, rank-16 adapters on the language model. A 10,000-image subset
(50,000 captions) keeps one epoch within a Kaggle session.

In [ ]:
!urducap train --model vlm --captions $CAPTIONS --images $IMAGES --epochs 1 --batch-size 8 --grad-accum 4 --lr 1e-4 --max-train-images 10000 --max-val-images 200 --out runs/vlm-lora
!urducap evaluate --run runs/vlm-lora --captions $CAPTIONS --images $IMAGES --batch-size 8 --beams 1 --out runs/vlm-lora

## 6. Results table

In [ ]:
import json, pathlib, pandas as pd
rows = {p.parent.name: json.loads(p.read_text()) for p in sorted(pathlib.Path("runs").glob("*/test_metrics.json"))}
table = pd.DataFrame(rows).T
print(table.to_markdown())
!cd runs && zip -q -r ../runs.zip . -x "*.pt"   # predictions and metrics, without weights